In [1]:
import pandas as pd
import os
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer

C:\Users\navid.hejazi\AppData\Local\anaconda3\envs\langchain_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%load_ext dotenv
%dotenv

In [3]:
files = pd.read_csv("course_section_descriptions.csv", encoding="ANSI")

In [4]:
files.columns

Index(['course_id', 'course_name', 'course_slug', 'course_description',
       'course_description_short', 'course_technology', 'course_topic',
       'course_instructor_quote', 'section_id', 'section_name',
       'section_description'],
      dtype='object')

In [5]:
files["unique_id"] = files["course_id"].astype(str) + files["section_id"].astype(str) 

In [6]:
files["metadata"] = files.apply(lambda row :{
    "course_name": row["course_name"],
    "section_name": row["section_name"],
    "section_description": row["section_description"],
}, axis=1)

In [7]:
files.columns

Index(['course_id', 'course_name', 'course_slug', 'course_description',
       'course_description_short', 'course_technology', 'course_topic',
       'course_instructor_quote', 'section_id', 'section_name',
       'section_description', 'unique_id', 'metadata'],
      dtype='object')

In [8]:
def create_embeddings(row):
    combined_text = f""" {row["course_name"]} {row["course_technology"]} {row["course_description"]}
    {row["section_name"]} {row["section_description"]}
    """
    return model.encode(combined_text, show_progress_bar = False)

In [9]:
model = SentenceTransformer("multi-qa-distilbert-cos-v1")

In [10]:
files["embedding"] = files.apply(create_embeddings, axis=1)

In [11]:
files.columns

Index(['course_id', 'course_name', 'course_slug', 'course_description',
       'course_description_short', 'course_technology', 'course_topic',
       'course_instructor_quote', 'section_id', 'section_name',
       'section_description', 'unique_id', 'metadata', 'embedding'],
      dtype='object')

In [12]:
load_dotenv(find_dotenv(), override = True)

True

In [13]:
pc=Pinecone(api_key = os.environ.get("PINECONE_API_KEY"), environment = os.environ.get("PINECONE_ENV"))

In [14]:
index_name = "bert"
dimension = 768
metric = "cosine"

In [26]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} succesfully deleted.")
else:
     print(f"{index_name} not in index list.")

bert succesfully deleted.


In [27]:
pc.create_index(
    name = index_name, 
    dimension = dimension, 
    metric = metric, 
    spec = ServerlessSpec(
        cloud = "aws", 
        region = "us-east-1")
    )

{
    "name": "bert",
    "metric": "cosine",
    "host": "bert-1bs5h5j.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "access-control-allow-origin": "*",
            "vary": "origin,access-control-request-method,access-control-request-headers",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",
          

In [17]:
index = pc.Index(index_name)

In [18]:
vectors_to_upsert = [(row["unique_id"], row["embedding"].tolist(), row["metadata"]) for index,row in files.iterrows()]
index.upsert(vectors=vectors_to_upsert)
print("Data succesfully upserted to pinecone index")

Data succesfully upserted to pinecone index


In [19]:
query = "regression in python"
query_embedding = model.encode(query, show_progress_bar = False).tolist()

In [20]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12,
    include_metadata = True
)

In [21]:
score_threshold = 0.4

In [22]:
# Assuming query_results are fetched and include metadata
for match in query_results["matches"]:
    if match["score"] >= score_threshold:
        course_details = match.get('metadata', {})
        course_name = course_details.get('course_name', "N/A")
        section_name = course_details.get('section_name', "N/A")
        section_description = course_details.get('section_description', "No description available")

        print(f"Matched item ID: {match['id']}, Score: {match['score']}")
        print(f"Course: {course_name} \nSection: {section_name} \nDescription: {section_description} \n---------------------\n")

Matched item ID: 37369, Score: 0.676096916
Course: Machine Learning in Python 
Section: Linear Regression with sklearn 
Description: While there are many libraries that can compute a regression model, the most numerically stable one is sklearn. It is also the preferred choice of many machine learning professionals. In this section, we implement all we know about regressions in this amazing library. 
---------------------

Matched item ID: 37368, Score: 0.606088638
Course: Machine Learning in Python 
Section: Linear Regression 
Description: In this part of the course, we will discuss what the course covers, why you need to learn advanced statistics, what’s the differences are with machine learning, and how to get the most out of this training. In this section, you will also expand on what you learned in our statistics training with additional concepts and will apply all the theory in Python. This section serves two purposes: 1) a useful refresher of regression, and 2) a great way to rei

## Weighted semantic search

In [23]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [25]:
files = pd.read_csv("course_section_descriptions.csv", encoding="ANSI")

In [24]:
# Define weights for different text components
weight_course_name = 5
weight_section_name = 3
weight_section_description = 2
weight_other = 1  # For other components like course technology and course description

In [28]:
def create_embeddings(row):
    # Encode individual components
    emb_course_name = model.encode(row['course_name'], show_progress_bar=False) * weight_course_name
    emb_section_name = model.encode(row['section_name'], show_progress_bar=False) * weight_section_name
    emb_section_description = model.encode(row['section_description'], show_progress_bar=False) * weight_section_description
    emb_course_tech = model.encode(row['course_technology'], show_progress_bar=False) * weight_other
    emb_course_desc = model.encode(row['course_description'], show_progress_bar=False) * weight_other

    # Combine embeddings by averaging them
    combined_embedding = (emb_course_name + emb_section_name + emb_section_description + emb_course_tech + emb_course_desc) / (weight_course_name + weight_section_name + weight_section_description + 2 * weight_other)
    return combined_embedding

In [29]:
# Initialize the model
model = SentenceTransformer('multi-qa-distilbert-cos-v1')

In [30]:
files['embedding'] = files.apply(create_embeddings, axis=1)

In [32]:
files["unique_id"] = files["course_id"].astype(str) + files["section_id"].astype(str) 

In [33]:
files["metadata"] = files.apply(lambda row :{
    "course_name": row["course_name"],
    "section_name": row["section_name"],
    "section_description": row["section_description"],
}, axis=1)

In [34]:
# Prepare the vectors for upserting
vectors_to_upsert = [(row['unique_id'], row['embedding'].tolist(), row['metadata']) for index, row in files.iterrows()]

In [35]:
# Upsert data
index.upsert(vectors=vectors_to_upsert)

print("Data successfully upserted to Pinecone index.")

Data successfully upserted to Pinecone index.


In [36]:
query = "regression in python"
query_embedding = model.encode(query, show_progress_bar = False).tolist()

In [37]:
query_results = index.query(
    vector = [query_embedding],
    top_k = 12,
    include_metadata = True
)

In [38]:
score_threshold = 0.4

In [39]:
# Assuming query_results are fetched and include metadata
for match in query_results["matches"]:
    if match["score"] >= score_threshold:
        course_details = match.get('metadata', {})
        course_name = course_details.get('course_name', "N/A")
        section_name = course_details.get('section_name', "N/A")
        section_description = course_details.get('section_description', "No description available")

        print(f"Matched item ID: {match['id']}, Score: {match['score']}")
        print(f"Course: {course_name} \nSection: {section_name} \nDescription: {section_description} \n---------------------\n")

Matched item ID: 37369, Score: 0.733964
Course: Machine Learning in Python 
Section: Linear Regression with sklearn 
Description: While there are many libraries that can compute a regression model, the most numerically stable one is sklearn. It is also the preferred choice of many machine learning professionals. In this section, we implement all we know about regressions in this amazing library. 
---------------------

Matched item ID: 37368, Score: 0.709501684
Course: Machine Learning in Python 
Section: Linear Regression 
Description: In this part of the course, we will discuss what the course covers, why you need to learn advanced statistics, what’s the differences are with machine learning, and how to get the most out of this training. In this section, you will also expand on what you learned in our statistics training with additional concepts and will apply all the theory in Python. This section serves two purposes: 1) a useful refresher of regression, and 2) a great way to reinfo